In [511]:
from pathlib import Path
import sys

sys.path.insert(0, "./")

from cancerfoundation.data.dataset import DatasetDir
from argparse import ArgumentParser
from bionemo.scdl.io.single_cell_collection import SingleCellCollection

import scanpy as sc
import pandas as pd
import json


GENE_ID = "_cf_gene_id"
CLS_TOKEN = "<cls>"
PAD_TOKEN = "<pad>"


def get_args():
    parser = ArgumentParser()
    parser.add_argument("--h5ad-path", type=Path)
    parser.add_argument("--data-path", type=Path)
    parser.add_argument("--vocab-path", type=Path, required=False)
    return parser.parse_args()


def _generate_vocab_from_h5ads(
    h5ads: Path, cls_token: str, pad_token: str
) -> dict[str, int]:
    genes = set()
    for path in h5ads.iterdir():
        if not path.name.endswith(".h5ad"):
            continue
        var_names = sc.read_h5ad(path, backed="r").var_names
        genes.update(var_names)
    vocab = {gene: i for i, gene in enumerate([cls_token, pad_token] + list(genes))}
    return vocab


def _save_vocab_to_dir(vocab: dict[str, int], data_dir: DatasetDir) -> None:
    with open(data_dir.vocab_path, "w") as f:
        json.dump(vocab, f)


def _add_gene_id_to_h5ads(h5ads: Path, vocab: dict[str, int], data_path: Path) -> None:
    (data_path / "h5ads").mkdir()
    for path in sorted(h5ads.iterdir()):
        if not path.name.endswith(".h5ad"):
            continue
        adata = sc.read_h5ad(path, backed="r")
        print(adata)
        adata.var[GENE_ID] = adata.var_names.map(vocab)
        adata.write_h5ad(data_path / "h5ads" / path.name)


def convert_columns_to_categorical_with_mapping(df):
    df_copy = df.copy()

    category_mappings = {}

    df_categorical = pd.DataFrame(index=df.index)

    for column in df.columns:
        df_copy[column] = df_copy[column].astype("category")

        category_mappings[column] = dict(
            zip(
                df_copy[column].cat.categories,
                range(len(df_copy[column].cat.categories)),
            )
        )

        df_categorical[column] = df_copy[column].cat.codes

    return df_categorical, category_mappings

In [ ]:
h5ad_path = Path("./DATA/medium/raw_data/train")
h5ad_path2 = Path("./DATA/medium/qqq/train")

arg_data_path = Path("./DATA/medium/processed_dataqq/train")
arg_data_path2 = Path("./DATA/medium/processed_dataqqq/train")
data_path = DatasetDir(arg_data_path)
data_path2 = DatasetDir(arg_data_path2)
data_path.mkdir()
data_path2.mkdir()

In [ ]:
vocab = _generate_vocab_from_h5ads(Path("./DATA/medium/raw_data/train"), CLS_TOKEN, PAD_TOKEN)
vocab2 = _generate_vocab_from_h5ads(Path("./DATA/medium/qqq/train"), CLS_TOKEN, PAD_TOKEN)

In [432]:
print(vocab)
print(vocab2)

{'<cls>': 0, '<pad>': 1, 'HGNC:32292': 2, 'HGNC:13595': 3, 'HGNC:17967': 4, 'HGNC:4564': 5, 'HGNC:11652': 6, 'HGNC:24934': 7, 'HGNC:9806': 8, 'HGNC:19006': 9, 'HGNC:21708': 10, 'HGNC:53263': 11, 'HGNC:4787': 12, 'HGNC:15710': 13, 'HGNC:5984': 14, 'HGNC:50656': 15, 'HGNC:34418': 16, 'HGNC:24637': 17, 'HGNC:8952': 18, 'HGNC:26191': 19, 'HGNC:1769': 20, 'HGNC:53067': 21, 'HGNC:25771': 22, 'HGNC:13626': 23, 'HGNC:52566': 24, 'HGNC:25077': 25, 'HGNC:475': 26, 'HGNC:4595': 27, 'HGNC:32000': 28, 'HGNC:25555': 29, 'HGNC:52883': 30, 'HGNC:29423': 31, 'HGNC:21734': 32, 'HGNC:50547': 33, 'HGNC:37982': 34, 'HGNC:4440': 35, 'HGNC:5393': 36, 'HGNC:1004': 37, 'HGNC:26138': 38, 'HGNC:16188': 39, 'HGNC:32687': 40, 'HGNC:43673': 41, 'HGNC:15153': 42, 'HGNC:19695': 43, 'HGNC:346': 44, 'HGNC:16181': 45, 'HGNC:52422': 46, 'HGNC:16206': 47, 'HGNC:10420': 48, 'HGNC:20748': 49, 'HGNC:45050': 50, 'HGNC:19751': 51, 'HGNC:21528': 52, 'HGNC:1610': 53, 'HGNC:7172': 54, 'HGNC:25179': 55, 'HGNC:32399': 56, 'HGNC:294

In [433]:
print(list(vocab.values())[-10:])

[23545, 23546, 23547, 23548, 23549, 23550, 23551, 23552, 23553, 23554]


In [434]:
print(list(vocab2.values())[-10:])

[23457, 23458, 23459, 23460, 23461, 23462, 23463, 23464, 23465, 23466]


In [484]:
_save_vocab_to_dir(vocab, data_path)
_save_vocab_to_dir(vocab2, data_path2)

In [485]:
_add_gene_id_to_h5ads(h5ad_path, vocab, arg_data_path)
_add_gene_id_to_h5ads(h5ad_path2, vocab2, arg_data_path2)

AnnData object with n_obs × n_vars = 48941 × 18720 backed at 'DATA/medium/raw_data/train/Bassez_breast.h5ad'
    obs: 'sample', 'cell_type', 'cell_subtype', 'timepoint', 'nCount_RNA', 'nFeature_RNA', 'expansion', 'BC_type', 'cohort', 'complexity', 'patient', 'n_cells', 'technology', 'cancer_type', 'sex', 'age', 'smoking_status', 'PY', 'diagnosis_recurrence', 'disease_extent', 'AJCC_T', 'AJCC_N', 'AJCC_M', 'AJCC_stage', 'sample_primary_met', 'treated_naive', 'size', 'site', 'histology', 'mutation_hormonal_subtype', 'grade', 'KI67', 'recent_treatment', 'recent_treatment_response', 'time_elapsed_from_recent_treatment', 'prior_chemotherapy', 'chemotherapy_response', 'prior_targeted_rx', 'targeted_rx_response', 'prior_ICB', 'ICB_response', 'prior_chemoICB', 'chemoICB_response', 'prior_ET', 'ET_response', 'subsequent_treatment', 'subsequent_treatment_response', 'PFS_DFS', 'OS', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'S_score', 'G2M_score', 'phase'

In [486]:
memmaps = SingleCellCollection(data_path.data_dir / "tmp")
memmaps2 = SingleCellCollection(data_path2.data_dir / "tmp")

In [438]:
print(memmaps.fname_to_mmap)

{}


In [487]:
memmaps.load_h5ad_multi(arg_data_path / "h5ads", max_workers=12)

MODIFIED


In [488]:
memmaps2.load_h5ad_multi(arg_data_path2 / "h5ads", max_workers=12)

MODIFIED


In [357]:
print(memmaps.fname_to_mmap)
print(memmaps2.fname_to_mmap)

{PosixPath('DATA/medium/processed_dataqq/train/tmp/Bassez_breast'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff436fefa10>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Bi_kidney'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff3f0412960>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Biermann_skin'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff3f0412030>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Bischoff_lung'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff436feeb10>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Caron_hematologic'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff42d667ec0>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Chan_lung'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff42d667560>, PosixPath('DATA/medium/processe

In [368]:
memmaps

In [179]:
columns = ["sample", "cancer_type", "technology", "tissue"]
obs_list = []
for i, fname in enumerate(memmaps.fname_to_mmap.keys()):
    print(f"Processing {i + 1}/{len(memmaps.fname_to_mmap)}: {fname.name}")
    adata = sc.read_h5ad(h5ad_path / (fname.name + ".h5ad"), backed="r")

    # Extract tissue name from the filename (e.g., '..._kidney' -> 'kidney')
    tissue_name = fname.name.split("_")[1]

    # Create the 'tissue' column and assign the extracted name to all cells
    adata.obs["tissue"] = tissue_name

    obs_list.append(adata.obs[columns])


Processing 1/10: Bassez_breast
Processing 2/10: Bi_kidney
Processing 3/10: Biermann_skin
Processing 4/10: Bischoff_lung
Processing 5/10: Caron_hematologic
Processing 6/10: Chan_lung
Processing 7/10: Chen_head-and-neck
Processing 8/10: Chen_prostate
Processing 9/10: Choudhury_brain
Processing 10/10: Chung_breast


In [180]:
columns = ["sample", "cancer_type", "technology", "tissue"]
obs_list2 = []
for i, fname in enumerate(memmaps2.fname_to_mmap.keys()):
    print(f"Processing {i + 1}/{len(memmaps2.fname_to_mmap)}: {fname.name}")
    adata = sc.read_h5ad(h5ad_path2 / (fname.name + ".h5ad"), backed="r")

    # Extract tissue name from the filename (e.g., '..._kidney' -> 'kidney')
    tissue_name = fname.name.split("_")[1]

    # Create the 'tissue' column and assign the extracted name to all cells
    adata.obs["tissue"] = tissue_name

    obs_list2.append(adata.obs[columns])


Processing 1/9: Bassez_breast
Processing 2/9: Bi_kidney
Processing 3/9: Biermann_skin
Processing 4/9: Bischoff_lung
Processing 5/9: Caron_hematologic
Processing 6/9: Chan_lung
Processing 7/9: Chen_head-and-neck
Processing 8/9: Chen_prostate
Processing 9/9: Choudhury_brain


In [183]:
print(obs_list)

[                                   sample    cancer_type technology  tissue
cell_name                                                                  
BIOKEY_1_On_AAACCTGGTCATACTG-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACCTGTCACGATGT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGCATCCCACT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGTCACCTTAT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAAGATGAGGTGACCA-1   BIOKEY_1  Breast Cancer        10X  breast
...                                   ...            ...        ...     ...
BIOKEY_9_Pre_TTTCCTCTCGAGGTAG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGGTTCACTGAAGG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAAGAATTGTG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAGTTCCGTCT-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCATCAGTTCGA-1  BIOKEY_9  Breast Cancer        10X  breast

[48941 row

In [184]:
print(obs_list2)

[                                   sample    cancer_type technology  tissue
cell_name                                                                  
BIOKEY_1_On_AAACCTGGTCATACTG-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACCTGTCACGATGT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGCATCCCACT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGTCACCTTAT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAAGATGAGGTGACCA-1   BIOKEY_1  Breast Cancer        10X  breast
...                                   ...            ...        ...     ...
BIOKEY_9_Pre_TTTCCTCTCGAGGTAG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGGTTCACTGAAGG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAAGAATTGTG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAGTTCCGTCT-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCATCAGTTCGA-1  BIOKEY_9  Breast Cancer        10X  breast

[48941 row

In [185]:
obs = pd.concat(obs_list)
obs2 = pd.concat(obs_list2)

In [190]:
print(obs)

                                  sample    cancer_type technology  tissue
cell_name                                                                 
BIOKEY_1_On_AAACCTGGTCATACTG-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACCTGTCACGATGT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGCATCCCACT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGTCACCTTAT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAAGATGAGGTGACCA-1  BIOKEY_1  Breast Cancer        10X  breast
...                                  ...            ...        ...     ...
BC05_92                             BC05  Breast Cancer  SmartSeq2  breast
BC05_93                             BC05  Breast Cancer  SmartSeq2  breast
BC05_94                             BC05  Breast Cancer  SmartSeq2  breast
BC05_95                             BC05  Breast Cancer  SmartSeq2  breast
BC05_96                             BC05  Breast Cancer  SmartSeq2  breast

[287398 rows x 4 columns

In [191]:
print(obs2)

                                  sample    cancer_type technology  tissue
cell_name                                                                 
BIOKEY_1_On_AAACCTGGTCATACTG-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACCTGTCACGATGT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGCATCCCACT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGTCACCTTAT-1  BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAAGATGAGGTGACCA-1  BIOKEY_1  Breast Cancer        10X  breast
...                                  ...            ...        ...     ...
MSC6-BTI_TTTGTTGCAGTTGTTG       MSC6-BTI     Meningioma        10x   brain
MSC6-BTI_TTTGTTGGTCAGACGA       MSC6-BTI     Meningioma        10x   brain
MSC6-BTI_TTTGTTGTCAAGGTGG       MSC6-BTI     Meningioma        10x   brain
MSC6-BTI_TTTGTTGTCGACATAC       MSC6-BTI     Meningioma        10x   brain
MSC6-BTI_TTTGTTGTCTGCTTAT       MSC6-BTI     Meningioma        10x   brain

[287171 rows x 4 columns

In [193]:
print(obs.tissue.unique())

['breast' 'kidney' 'skin' 'lung' 'hematologic' 'head-and-neck' 'prostate'
 'brain']


In [192]:
print(obs2.tissue.unique())

['breast' 'kidney' 'skin' 'lung' 'hematologic' 'head-and-neck' 'prostate'
 'brain']


In [194]:
obs, mapping = convert_columns_to_categorical_with_mapping(obs)

In [195]:
obs2, mapping2 = convert_columns_to_categorical_with_mapping(obs2)

In [201]:
print(obs.tissue.unique())

[1 4 7 5 3 2 6 0]


In [200]:
print(obs2.tissue.unique())

[1 4 7 5 3 2 6 0]


In [ ]:
print(obs.to_parquet(data_path.obs_path))


None


In [206]:
print(obs.to_parquet(data_path2.obs_path))

None


In [104]:
print(obs_list[0])

                                   sample    cancer_type technology  tissue
cell_name                                                                  
BIOKEY_1_On_AAACCTGGTCATACTG-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACCTGTCACGATGT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGCATCCCACT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAACGGGTCACCTTAT-1   BIOKEY_1  Breast Cancer        10X  breast
BIOKEY_1_On_AAAGATGAGGTGACCA-1   BIOKEY_1  Breast Cancer        10X  breast
...                                   ...            ...        ...     ...
BIOKEY_9_Pre_TTTCCTCTCGAGGTAG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGGTTCACTGAAGG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAAGAATTGTG-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCAGTTCCGTCT-1  BIOKEY_9  Breast Cancer        10X  breast
BIOKEY_9_Pre_TTTGTCATCAGTTCGA-1  BIOKEY_9  Breast Cancer        10X  breast

[48941 rows

In [204]:
print(mapping)

{'sample': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 11: 9, 12: 10, 13: 11, 'BC02': 12, 'BC04': 13, 'BC05': 14, 'BC07': 15, 'BC07LN': 16, 'BIOKEY_1': 17, 'BIOKEY_10': 18, 'BIOKEY_11': 19, 'BIOKEY_12': 20, 'BIOKEY_13': 21, 'BIOKEY_14': 22, 'BIOKEY_15': 23, 'BIOKEY_16': 24, 'BIOKEY_17': 25, 'BIOKEY_18': 26, 'BIOKEY_19': 27, 'BIOKEY_2': 28, 'BIOKEY_20': 29, 'BIOKEY_21': 30, 'BIOKEY_22': 31, 'BIOKEY_23': 32, 'BIOKEY_24': 33, 'BIOKEY_25': 34, 'BIOKEY_26': 35, 'BIOKEY_27': 36, 'BIOKEY_28': 37, 'BIOKEY_29': 38, 'BIOKEY_3': 39, 'BIOKEY_30': 40, 'BIOKEY_31': 41, 'BIOKEY_32': 42, 'BIOKEY_33': 43, 'BIOKEY_35': 44, 'BIOKEY_36': 45, 'BIOKEY_37': 46, 'BIOKEY_38': 47, 'BIOKEY_39': 48, 'BIOKEY_4': 49, 'BIOKEY_40': 50, 'BIOKEY_41': 51, 'BIOKEY_42': 52, 'BIOKEY_5': 53, 'BIOKEY_6': 54, 'BIOKEY_7': 55, 'BIOKEY_8': 56, 'BIOKEY_9': 57, 'ETV6.RUNX1.1': 58, 'ETV6.RUNX1.2': 59, 'ETV6.RUNX1.3': 60, 'ETV6.RUNX1.4': 61, 'HHD.1': 62, 'HHD.2': 63, 'MBM01_sc': 64, 'MBM02_sc': 65, 'MBM03_sc': 66, 'MBM04_

In [205]:
print(mapping2)

{'sample': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 11: 9, 12: 10, 13: 11, 'BIOKEY_1': 12, 'BIOKEY_10': 13, 'BIOKEY_11': 14, 'BIOKEY_12': 15, 'BIOKEY_13': 16, 'BIOKEY_14': 17, 'BIOKEY_15': 18, 'BIOKEY_16': 19, 'BIOKEY_17': 20, 'BIOKEY_18': 21, 'BIOKEY_19': 22, 'BIOKEY_2': 23, 'BIOKEY_20': 24, 'BIOKEY_21': 25, 'BIOKEY_22': 26, 'BIOKEY_23': 27, 'BIOKEY_24': 28, 'BIOKEY_25': 29, 'BIOKEY_26': 30, 'BIOKEY_27': 31, 'BIOKEY_28': 32, 'BIOKEY_29': 33, 'BIOKEY_3': 34, 'BIOKEY_30': 35, 'BIOKEY_31': 36, 'BIOKEY_32': 37, 'BIOKEY_33': 38, 'BIOKEY_35': 39, 'BIOKEY_36': 40, 'BIOKEY_37': 41, 'BIOKEY_38': 42, 'BIOKEY_39': 43, 'BIOKEY_4': 44, 'BIOKEY_40': 45, 'BIOKEY_41': 46, 'BIOKEY_42': 47, 'BIOKEY_5': 48, 'BIOKEY_6': 49, 'BIOKEY_7': 50, 'BIOKEY_8': 51, 'BIOKEY_9': 52, 'ETV6.RUNX1.1': 53, 'ETV6.RUNX1.2': 54, 'ETV6.RUNX1.3': 55, 'ETV6.RUNX1.4': 56, 'HHD.1': 57, 'HHD.2': 58, 'MBM01_sc': 59, 'MBM02_sc': 60, 'MBM03_sc': 61, 'MBM04_sc': 62, 'MBM05_sc': 63, 'MBM05_sn': 64, 'MBM06_sn': 65, 'MBM0

In [ ]:
print(memmaps)

In [489]:
memmaps.flatten(data_path.memmap_path, destroy_on_copy=False)

In [359]:
print(memmaps.fname_to_mmap)

{PosixPath('DATA/medium/processed_dataqq/train/tmp/Bassez_breast'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff436fefa10>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Bi_kidney'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff3f0412960>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Biermann_skin'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff3f0412030>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Bischoff_lung'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff436feeb10>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Caron_hematologic'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff42d667ec0>, PosixPath('DATA/medium/processed_dataqq/train/tmp/Chan_lung'): <bionemo.scdl.io.single_cell_memmap_dataset.SingleCellMemMapDataset object at 0x7ff42d667560>, PosixPath('DATA/medium/processe

In [208]:
print(memmaps2)

In [490]:
memmaps2.flatten(data_path2.memmap_path, destroy_on_copy=False)

In [212]:
print(memmaps2)

In [493]:
from bionemo.scdl.io.single_cell_memmap_dataset import SingleCellMemMapDataset
mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq/train/mem.map")
print(mm._feature_index.number_vars_at_row(48941))
mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq2/train/mem.map")
print(mm._feature_index.number_vars_at_row(48941))

18720
19865


In [510]:
combined.obs['sample']

cell_name
BIOKEY_1_On_AAACCTGGTCATACTG-1__f0    BIOKEY_1
BIOKEY_1_On_AAACCTGTCACGATGT-1__f0    BIOKEY_1
BIOKEY_1_On_AAACGGGCATCCCACT-1__f0    BIOKEY_1
BIOKEY_1_On_AAACGGGTCACCTTAT-1__f0    BIOKEY_1
BIOKEY_1_On_AAAGATGAGGTGACCA-1__f0    BIOKEY_1
                                        ...   
BC05_92__f9                               BC05
BC05_93__f9                               BC05
BC05_94__f9                               BC05
BC05_95__f9                               BC05
BC05_96__f9                               BC05
Name: sample, Length: 287398, dtype: object

In [505]:
# Build one combined AnnData with a union of genes
import scanpy as sc
from pathlib import Path

src = Path("./DATA/medium/raw_data/train")
paths = sorted(p for p in src.iterdir() if p.name.endswith(".h5ad"))

adatas = [sc.read_h5ad(p, backed=None) for p in paths]  # read into memory
# Optional: ensure obs_names are unique per file
for i, ad in enumerate(adatas):
    ad.obs_names = ad.obs_names.astype(str) + f"__f{i}"

combined = adatas[0].concatenate(*adatas[1:], join="outer", index_unique=None)  # union of vars

combined_path = Path("./DATA/medium/combined/train_union.h5ad")
combined.write_h5ad(combined_path)

# Now create a single SC memmap directly from the combined file (no flatten)
from bionemo.scdl.io.single_cell_memmap_dataset import SingleCellMemMapDataset
mm = SingleCellMemMapDataset(data_path="DATA/medium/processed_data_union/train/mem.map",
                             h5ad_path=str(combined_path))


/tmp/ipykernel_21194/2529249770.py:13: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = adatas[0].concatenate(*adatas[1:], join="outer", index_unique=None)  # union of vars
/usr/local/lib/python3.12/dist-packages/anndata/_core/merge.py:1667: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/usr/local/lib/python3.12/dist-packages/anndata/_core/merge.py:1667: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when

TypeError: Can't implicitly convert non-string objects to strings

In [502]:
orig = mm._feature_index.number_vars_at_row

def nvars_safe(i):
    print("QQQ")
    n = orig(i)
    s, e = mm.row_index[i], mm.row_index[i+1]
    if e > s:
        mx = int(mm.col_index[s:e].max())
        if mx >= n:
            return mx + 1           # trust CSR over RFI at boundaries
    return n

mm._feature_index.number_vars_at_row = nvars_safe

In [501]:
mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq/train/mem.map")
print(mm._feature_index.number_vars_at_row(48941))

18720


In [503]:
print(mm._feature_index.number_vars_at_row(48941))

QQQ
19863


In [466]:
import numpy as np, json
from bionemo.scdl.io.single_cell_memmap_dataset import SingleCellMemMapDataset

mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq/train/mem.map")

# 1) What block sizes does the RFI think it has?
print("column_dims:", mm._feature_index.column_dims())  # expect [18720, 19865, 22204, 22673, ...]
try:
    print("row_offsets:", mm._feature_index.row_offsets)  # starts [0, 48941, 48941+4615, ...] if bug
except AttributeError:
    pass

# 2) Boundary sanity: just around the first boundary
for j in [48940, 48941, 48942, 53555, 53556, 53557]:
    print(j, mm._feature_index.number_vars_at_row(j))

def row_bounds_ok(i):
    start = mm.row_index[i]
    end   = mm.row_index[i+1]
    cols  = mm.col_index[start:end]
    ncols = mm._feature_index.number_vars_at_row(i)
    bad   = cols[(cols < 0) | (cols >= ncols)]
    return bad, cols, ncols

for i in range(53675):
    bad, cols, ncols = row_bounds_ok(i)
    
    if bad.size:
        print(f"ROW {i}: ncols={ncols}, max(col)={cols.max()}, min(col)={cols.min()}, first_bad={bad[0]}")
        break
else:
    print("All rows OK")

column_dims: [18720, 18720, 19865, 22204, 22673, 22387, 20234, 22710, 19165, 22673, 22141]
48940 18720
48941 18720
48942 18720
53555 18720
53556 19865
53557 19865
ROW 48941: ncols=18720, max(col)=19862, min(col)=6, first_bad=18721


In [467]:
import numpy as np, json
from bionemo.scdl.io.single_cell_memmap_dataset import SingleCellMemMapDataset

mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq2/train/mem.map")

# 1) What block sizes does the RFI think it has?
print("column_dims:", mm._feature_index.column_dims())  # expect [18720, 19865, 22204, 22673, ...]
try:
    print("row_offsets:", mm._feature_index.row_offsets)  # starts [0, 48941, 48941+4615, ...] if bug
except AttributeError:
    pass

# 2) Boundary sanity: just around the first boundary
for j in [48940, 48941, 48942, 53555, 53556, 53557]:
    print(j, mm._feature_index.number_vars_at_row(j))
    
def row_bounds_ok(i):
    start = mm.row_index[i]
    end   = mm.row_index[i+1]
    cols  = mm.col_index[start:end]
    ncols = mm._feature_index.number_vars_at_row(i)
    bad   = cols[(cols < 0) | (cols >= ncols)]
    return bad, cols, ncols

for i in range(50000):
    bad, cols, ncols = row_bounds_ok(i)
    if bad.size:
        print(f"ROW {i}: ncols={ncols}, max(col)={cols.max()}, min(col)={cols.min()}, first_bad={bad[0]}")
        break
else:
    print("All rows OK")
print(ncols)

column_dims: [18720, 19865, 22204, 22673, 22387, 20234, 22710, 19165, 22673]
48940 18720
48941 19865
48942 19865
53555 19865
53556 22204
53557 22204
All rows OK
19865


In [214]:
print(mm)

In [331]:
mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq/train/mem.map")
start = mm.row_index[0]
end   = mm.row_index[1]
cols  = mm.col_index[start:end]
ncols = mm._feature_index.number_vars_at_row(i)
print(cols.max())
print(ncols)

18686
18720


In [332]:
mm = SingleCellMemMapDataset("DATA/medium/processed_dataqq2/train/mem.map")
start = mm.row_index[0]
end   = mm.row_index[1]
cols  = mm.col_index[start:end]
ncols = mm._feature_index.number_vars_at_row(i)
print(cols.max())
print(ncols)

18686
19865
